# Reachability: proposal hard Sokoban families

Offline walkthrough of the 32-shot exploratory pilot. For each family the frozen cohort has 50 layouts; the first 20 are planned for each model and offset. `k` counts actions in the witness, not shortest-path distance. A hit requires exact target state including the player. Missing or unresolved trials are not failures.

The plots below read the replay-verified N20 export. They never substitute fixtures when results are absent.

In [ ]:
from pathlib import Path
import json, csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from carl.icl.boards import from_rows, to_rows, state_hash
from carl.icl.data_sources.sokoban import environment

def repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / 'carl').is_dir() and (p / 'analysis/icl_reachability').is_dir(): return p
    raise RuntimeError('Run this notebook inside the CaRL repository.')
repo = repo_root()
run = repo / 'analysis/icl_reachability/figure1_proposal_hard_v1'
recovery_results = repo / 'analysis/icl_reachability/figure1_proposal_hard_recovery_v1/pilot20_export'
recovery_v2 = repo / 'analysis/icl_reachability/figure1_proposal_hard_recovery_v2/pilot20_export'
recovery_v3 = repo / 'analysis/icl_reachability/figure1_proposal_hard_recovery_v3/pilot20_export'
recovery_v4 = repo / 'analysis/icl_reachability/figure1_proposal_hard_recovery_v4/pilot20_export'
recovery_v5 = repo / 'analysis/icl_reachability/figure1_proposal_hard_recovery_v5/pilot20_export'
results = next((p for p in (recovery_v5, recovery_v4, recovery_v3, recovery_v2, recovery_results, run / 'pilot20_export') if (p / 'verification.json').is_file()), run / 'pilot20_export')
print('Results:', results)
for family in (12, 32):
    m=json.loads((repo/f'data/icl/reachability_proposal_hard_{family}_v1/manifest.json').read_text())
    print(m['family'], 'layouts', m['main_selected'], 'K', m['offsets'], 'training:', m['training'])

## Frozen source states
These are the actual `s_i` states, in cohort order. The first 20 are in the pilot. Each has a common source across all offsets and models.

In [ ]:
for family in (12,32):
    p=run/f'previews/anchors_50_{family}.png'
    if p.exists(): display(Image.open(p))
    else: print('Missing preview:', p)

## Reachability results
Circles denote complete 20-layout cells; crosses denote available observations only. Confidence intervals resample layouts. An all-zero bootstrap does not prove zero probability.

In [ ]:
summary_path = results/'summary_by_k_model.csv'
if not summary_path.exists():
    print('Full paired export not yet available. Showing action/RNG-replayed Luna N20 if present.')
    interim=run/'luna_n20_comparison.png'
    if interim.exists(): display(Image.open(interim))
else:
    with summary_path.open() as f: summary=list(csv.DictReader(f))
    for row in summary: print(row)
    plot=results/'reachability.png'
    if plot.exists(): display(Image.open(plot))
    else: print('Missing plot:', plot)
    inspired=results/'figure1_style_empirical.png'
    if inspired.exists():
        print('Figure 1 style: empirical measurements and partial-data bounds')
        display(Image.open(inspired))
completed12=repo/'analysis/icl_reachability/figure1_proposal_hard_recovery_v1/complete_12_comparison/comparison.png'
if completed12.is_file():
    print('12×12×10: both models, complete N20 (action/RNG replay)')
    display(Image.open(completed12))


## A recorded Luna success
The actions are replayed using the exact CaRL environment; each intermediate hash must match the recorded trace. The prompt shown below is the actual cached payload, including demonstrations, for the first action.

In [ ]:
paths=sorted((run/'study_12/trials').glob('*.json'))
matches=[]
for path in paths:
    r=json.loads(path.read_text())
    if r.get('status')=='complete' and r.get('exact_hit') and r['identity']['arm']=='luna': matches.append((path,r))
if not matches:
    print('No recorded Luna success yet.')
else:
    path,r=matches[0]; ident=r['identity']; print('Trial:',path.name,'k=',ident['k'],'first hit=',r['first_hit'])
    data=repo/'data/icl/reachability_proposal_hard_12_v1'
    pairs=json.loads((data/'core_pairs.json').read_text())['pairs']
    states=json.loads((data/'states.json').read_text())['states']
    pair=next(p for p in pairs if p['pair_id']==ident['pair'])
    current=from_rows(states[pair['source_hash']]); target=from_rows(states[pair['target_hash']]); env=environment(current)
    tile_names={'#':'wall.png','-':'floor.png','.':'box_target.png','$':'box.png','*':'box_on_target.png','@':'player.png','+':'player_on_target.png'}
    tiles={k:np.asarray(Image.open(repo/'carl/environment/sokoban/surface/16x16pixels'/v).convert('RGB')) for k,v in tile_names.items()}
    def show_board(board,title):
        pixels=np.concatenate([np.concatenate([tiles[ch] for ch in row],axis=1) for row in to_rows(board)],axis=0)
        print(title); display(Image.fromarray(pixels).resize((384,384), Image.Resampling.NEAREST))
    show_board(current, 'Source s_i'); show_board(target, 'Target s_i+k')
    names=['UP','DOWN','LEFT','RIGHT']
    for i,step in enumerate(r['steps'],1):
        assert state_hash(current)==step['before_hash']
        current=np.asarray(env.next_state(current,step['action']))
        assert state_hash(current)==step['after_hash']
        print(i,names[step['action']],step['probabilities']); show_board(current,f'Action {i}: {names[step["action"]]}')
    assert state_hash(current)==pair['target_hash']
    cache=run/'cache_12/live'/f"{r['requests'][0]['request_hash']}.json"
    if cache.exists():
        payload=json.loads(cache.read_text())['request']['payload']
        print('Actual request payload (32 demonstrations included):')
        print(json.dumps(payload,indent=2,ensure_ascii=False))
    else: print('Missing request cache:',cache)